<a href="https://colab.research.google.com/github/Shubham-Chavan-7/Sales-Analysis/blob/main/EV_Population.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import warnings
from datetime import datetime

import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# forecasting
from statsmodels.tsa.arima.model import ARIMA

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")


In [6]:
EXCEL_PATH = "/content/Electric_Vehicle_Population_Data_Excel_sheet.xlsx"
OUTPUT_DIR = "./outputs"
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
CSVS_DIR = os.path.join(OUTPUT_DIR, "csvs")
REPORT_PATH = os.path.join(OUTPUT_DIR, "report_summary.txt")

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(CSVS_DIR, exist_ok=True)

In [9]:
def safe_read_excel(path):
    """Try to read the excel; if multiple sheets exist, choose the first non-empty."""
    xls = pd.ExcelFile(path)
    for sheet in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet, engine="openpyxl")
        if df.shape[0] > 0:
            return df
    # fallback
    return pd.read_excel(path, engine="openpyxl")


def coerce_numeric(series, col_name=None):
    """Coerce to numeric, strip commas and non-numeric; return float dtype and count of coerced values."""
    s = series.copy().astype(str).str.replace(",", "").str.replace(r"[^\d\.\-]", "", regex=True)
    coerced = pd.to_numeric(s, errors="coerce")
    n_coerced = coerced.isna().sum()
    return coerced, n_coerced

In [10]:
print("Loading data from:", EXCEL_PATH)
df_raw = safe_read_excel(EXCEL_PATH)
print(f"Original rows: {len(df_raw)}, columns: {len(df_raw.columns)}")

# make a working copy
df = df_raw.copy()


Loading data from: /content/Electric_Vehicle_Population_Data_Excel_sheet.xlsx
Original rows: 177866, columns: 17


In [11]:
def normalize_colname(c):
    return c.strip().lower().replace(" ", "_").replace("-", "_")

df.columns = [normalize_colname(c) for c in df.columns]


In [12]:
col_map = {}
if "model_year" not in df.columns:
    for candidate in ["modelyear", "model year", "model_year", "model-year"]:
        if candidate in df.columns:
            col_map[candidate] = "model_year"
if "make" not in df.columns:
    for candidate in ["make", "manufacturer"]:
        if candidate in df.columns:
            col_map[candidate] = "make"
if "model" not in df.columns:
    for candidate in ["model"]:
        if candidate in df.columns:
            col_map[candidate] = "model"
if "electric_range" not in df.columns:
    for candidate in ["electric_range", "electric range", "range"]:
        if candidate in df.columns:
            col_map[candidate] = "electric_range"
if "state" not in df.columns:
    for candidate in ["state", "st", "province"]:
        if candidate in df.columns:
            col_map[candidate] = "state"
if "city" not in df.columns:
    for candidate in ["city", "town"]:
        if candidate in df.columns:
            col_map[candidate] = "city"
if "county" not in df.columns:
    for candidate in ["county"]:
        if candidate in df.columns:
            col_map[candidate] = "county"
if "base_msrp" not in df.columns:
    for candidate in ["base_msrp", "msrp", "base price", "base_price"]:
        if candidate in df.columns:
            col_map[candidate] = "base_msrp"

# apply mapping
df = df.rename(columns=col_map)

In [45]:
cleaning_report = []
insights = []

In [ ]:
# Model Year: coerce to integer where possible
if "model_year" in df.columns:
    df["model_year"] = pd.to_numeric(df["model_year"], errors="coerce").astype("Int64")
    n_missing_my = df["model_year"].isna().sum()
    cleaning_report.append(f"model_year missing or invalid: {n_missing_my}")
else:
    cleaning_report.append("model_year column not found")


In [ ]:
# Electric range: coerce numeric
if "electric_range" in df.columns:
    df["electric_range_clean"], n_coerced = coerce_numeric(df["electric_range"], "electric_range")
    df["electric_range_clean"] = df["electric_range_clean"].astype("Float64")
    cleaning_report.append(f"electric_range coerced to numeric; non-convertible values: {n_coerced}")
else:
    cleaning_report.append("electric_range column not found")

In [ ]:
# base_msrp: numeric conversion
if "base_msrp" in df.columns:
    df["base_msrp_clean"], n_coerced_msrp = coerce_numeric(df["base_msrp"], "base_msrp")
    df["base_msrp_clean"] = df["base_msrp_clean"].astype("Float64")
    cleaning_report.append(f"base_msrp coerced; non-convertible: {n_coerced_msrp}")
else:
    cleaning_report.append("base_msrp column not found")


In [ ]:
# Make/Model: strip whitespace, uppercase make
if "make" in df.columns:
    df["make_clean"] = df["make"].astype(str).str.strip().str.title()
else:
    df["make_clean"] = np.nan

if "model" in df.columns:
    df["model_clean"] = df["model"].astype(str).str.strip()
else:
    df["model_clean"] = np.nan

In [ ]:
# State/city/county cleanup
for geo in ["state", "city", "county"]:
    if geo in df.columns:
        df[geo + "_clean"] = df[geo].astype(str).str.strip()
    else:
        df[geo + "_clean"] = np.nan

In [15]:
# Drop rows with no VIN AND no model_year AND no location: very sparse
initial_rows = len(df)

# Ensure make_clean and model_clean columns exist before dropping
# This is a defensive measure in case previous cleaning steps were not executed
if "make_clean" not in df.columns:
    if "make" in df.columns:
        df["make_clean"] = df["make"].astype(str).str.strip().str.title()
    else:
        df["make_clean"] = np.nan

if "model_clean" not in df.columns:
    if "model" in df.columns:
        df["model_clean"] = df["model"].astype(str).str.strip()
    else:
        df["model_clean"] = np.nan

df = df.dropna(subset=["model_year", "make_clean", "model_clean"], how="all")
dropped_rows = initial_rows - len(df)
cleaning_report.append(f"Rows dropped for having all key identifiers missing: {dropped_rows}")

In [16]:
# ---------- DESCRIPTIVE STATISTICS ----------
desc_nums = {}
if "electric_range_clean" in df.columns:
    desc_nums["electric_range"] = df["electric_range_clean"].describe().to_dict()
if "base_msrp_clean" in df.columns:
    desc_nums["base_msrp"] = df["base_msrp_clean"].describe().to_dict()

In [18]:
# Yearly counts
if "model_year" in df.columns:
    yearly_counts = (
        df.dropna(subset=["model_year"])
        .groupby("model_year")
        .size()
        .sort_index()
        .astype(int)
        .rename("registrations")
    )
else:
    yearly_counts = pd.Series(dtype=int, name="registrations")

In [19]:
# Save numeric summaries
with open(os.path.join(CSVS_DIR, "numeric_descriptives.txt"), "w") as f:
    f.write("Numeric column descriptive summaries\n")
    f.write(str(desc_nums))

yearly_counts.to_csv(os.path.join(CSVS_DIR, "yearly_registrations.csv"))

In [21]:
# Top makes and models
top_makes = df.groupby("make_clean").size().sort_values(ascending=False).head(20)
top_models = df.groupby(["make_clean", "model_clean"]).size().sort_values(ascending=False).head(30)
top_makes.to_csv(os.path.join(CSVS_DIR, "top_makes.csv"))
top_models.to_csv(os.path.join(CSVS_DIR, "top_models.csv"))

In [22]:
# Geographic counts (by state / county / city)
geo_cols = []
if "state_clean" in df.columns:
    state_counts = df.groupby("state_clean")["record_count"].sum().sort_values(ascending=False)
    state_counts.to_csv(os.path.join(CSVS_DIR, "state_counts.csv"))
    geo_cols.append("state")
if "county_clean" in df.columns:
    county_counts = df.groupby("county_clean")["record_count"].sum().sort_values(ascending=False)
    county_counts.to_csv(os.path.join(CSVS_DIR, "county_counts.csv"))
    geo_cols.append("county")
if "city_clean" in df.columns:
    city_counts = df.groupby("city_clean")["record_count"].sum().sort_values(ascending=False)
    city_counts.to_csv(os.path.join(CSVS_DIR, "city_counts.csv"))
    geo_cols.append("city")


In [23]:
# ---------- VISUALIZATIONS ----------
def savefig(fig, fname, dpi=150):
    path = os.path.join(PLOTS_DIR, fname)
    fig.savefig(path, bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    print("Saved plot:", path)


In [24]:
# 1) Yearly registrations line chart
if len(yearly_counts) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(yearly_counts.index.astype(int), yearly_counts.values, marker="o")
    ax.set_title("EV Registrations by Model Year")
    ax.set_xlabel("Model Year")
    ax.set_ylabel("Number of Registrations")
    savefig(fig, "yearly_registrations_line.png")

Saved plot: ./outputs/plots/yearly_registrations_line.png


In [25]:
# 2) Distribution: electric range histogram + boxplot
if "electric_range_clean" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(df["electric_range_clean"].dropna(), bins=30, kde=False, ax=ax)
    ax.set_title("Electric Range Distribution (miles)")
    ax.set_xlabel("Electric Range (miles)")
    savefig(fig, "electric_range_hist.png")

In [27]:
if "electric_range_clean" in df.columns:
    fig2, ax2 = plt.subplots(figsize=(8, 3))
    sns.boxplot(x=df["electric_range_clean"].dropna(), ax=ax2)
    ax2.set_title("Electric Range Boxplot")
    savefig(fig2, "electric_range_boxplot.png")

In [28]:
# 3) Top makes bar chart
if len(top_makes) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    top_makes.head(15).plot(kind="bar", ax=ax)
    ax.set_title("Top 15 EV Makes by Registration Count")
    ax.set_ylabel("Registrations")
    savefig(fig, "top_makes_bar.png")

Saved plot: ./outputs/plots/top_makes_bar.png


In [29]:
# 4) Scatter: base MSRP vs electric range (if both available)
if "base_msrp_clean" in df.columns and "electric_range_clean" in df.columns:
    subset = df.dropna(subset=["base_msrp_clean", "electric_range_clean"])
    if len(subset) > 10:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(subset["base_msrp_clean"], subset["electric_range_clean"], alpha=0.5)
        ax.set_xlabel("Base MSRP (USD)")
        ax.set_ylabel("Electric Range (miles)")
        ax.set_title("Base MSRP vs Electric Range")
        savefig(fig, "msrp_vs_range_scatter.png")

In [30]:
# 5) Heatmap by top states (bar chart since we may not have shapefiles)
if "state_clean" in df.columns:
    top_state_counts = state_counts.head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    top_state_counts.plot(kind="bar", ax=ax)
    ax.set_title("Top States by EV Registrations (Top 20)")
    savefig(fig, "top_states_bar.png")

In [31]:
# ---------- SIMPLE FORECAST (ARIMA) on yearly registrations ----------
forecast_results = {}
if len(yearly_counts) >= 5:
    # Prepare time series (index as int years)
    ts = yearly_counts.copy()
    ts.index = pd.to_datetime(ts.index.astype(int).astype(str) + "-01-01")
    ts = ts.asfreq("YS")  # yearly start
    ts = ts.fillna(0)


In [42]:
# Fit a simple ARIMA(p,d,q) — choose (1,1,1) as a reasonable default and catch fit errors
try:
    model = ARIMA(ts, order=(1, 1, 1))
    model_fit = model.fit()
    steps = 5  # forecast next 5 years
    forecast = model_fit.get_forecast(steps=steps)
    fc_mean = forecast.predicted_mean
    fc_ci = forecast.conf_int()
    forecast_results = {
        "mean": fc_mean.to_dict(),
        "lower_ci": fc_ci.iloc[:, 0].to_dict(),
        "upper_ci": fc_ci.iloc[:, 1].to_dict(),
        "status": "success" # Set status to success here
    }
except Exception as e:
    forecast_results["error"] = f"ARIMA forecast failed: {e}"
    forecast_results["status"] = "failed"
    print(f"ARIMA forecast failed: {e}")

In [39]:
# Save forecast to CSV
fc_df = pd.DataFrame({
    "forecast_mean": fc_mean.values,
    "lower_ci": fc_ci.iloc[:, 0].values,
    "upper_ci": fc_ci.iloc[:, 1].values
}, index=fc_mean.index)
fc_df.to_csv(os.path.join(CSVS_DIR, "yearly_registrations_forecast.csv"))

In [43]:
if forecast_results.get("status") == "success":
    # Plot historical + forecast
    fig, ax = plt.subplots(figsize=(10, 6))
    ts.plot(ax=ax, label="Historical")
    fc_mean.plot(ax=ax, label="Forecast (mean)")
    ax.fill_between(fc_df.index, fc_df["lower_ci"], fc_df["upper_ci"], color="gray", alpha=0.3)
    ax.set_title("Historical EV Registrations and 5-year ARIMA Forecast")
    ax.set_xlabel("Year")
    ax.set_ylabel("Registrations")
    ax.legend()
    savefig(fig, "yearly_registrations_forecast.png")
else:
    print(f"ARIMA forecast was not successful or skipped: {forecast_results.get('error', 'Not enough data')}")

Saved plot: ./outputs/plots/yearly_registrations_forecast.png


In [47]:
# Growth trend: compute CAGR over available years if possible
if len(yearly_counts) >= 2:
    years = yearly_counts.index.astype(int).values
    first_year, last_year = years[0], years[-1]
    first_val = int(yearly_counts.iloc[0])
    last_val = int(yearly_counts.iloc[-1])
    if first_val > 0:
        years_diff = last_year - first_year
        cagr = (last_val / first_val) ** (1 / years_diff) - 1 if years_diff > 0 else np.nan
        insights.append(f"From {first_year} to {last_year}, registrations grew from {first_val} to {last_val} (approx CAGR = {cagr:.2%}).")
    else:
        insights.append(f"Registrations in {first_year} is zero or missing; cannot compute CAGR.")

In [50]:
# Top makes insight
if len(top_makes) > 0:
    top_make = top_makes.idxmax()
    top_make_count = int(top_makes.max())
    insights.append(f"Top make: {top_make} with {top_make_count} registrations (top of dataset).")

# Electric range median
if "electric_range_clean" in df.columns:
    median_range = df["electric_range_clean"].median(skipna=True)
    insights.append(f"Median electric range across vehicles: {median_range:.0f} miles (where available).")

# MSRP vs Range correlation
if "base_msrp_clean" in df.columns and "electric_range_clean" in df.columns:
    corr = df[["base_msrp_clean", "electric_range_clean"]].dropna().corr().iloc[0, 1]
    insights.append(f"Pearson correlation between Base MSRP and Electric Range: {corr:.2f} (positive implies higher price tends to have longer range).")

# Forecast interpretation
if forecast_results.get("status") == "success":
    # The fc_df DataFrame is already available from previous cells
    first_fc_year = fc_df.index[0].year
    last_fc_year = fc_df.index[-1].year
    insights.append(f"ARIMA 5-year forecast produced for years {first_fc_year}–{last_fc_year}. See CSV for forecast numbers.")
elif forecast_results.get("status") == "skipped":
    insights.append("Forecasting skipped: insufficient yearly data points.")
else:
    insights.append(f"Forecasting failed: {forecast_results.get('error', 'unknown error')}")